In [54]:
import mne
import pandas as pd
import pyedflib
from copy import deepcopy
from typing import Final
from scipy import stats

In [ ]:
raw = mne.io.read_raw_edf("data/eeg_test.edf", preload=True)
ecg_like = [ch for ch in raw.ch_names if "ECG" in ch.upper()]
if ecg_like:
    raw.drop_channels(ecg_like)
numpoint: Final[int] = 50000
start_phys = 0
data = raw.get_data()
data = data[:, start_phys:numpoint]
channel_names = raw.ch_names
df = pd.DataFrame(data.T, columns=raw.ch_names)
df.T.to_csv("data/eeg_test_converted.csv", index=False, header=False)

In [56]:
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
# print(raw)
# print(raw.info)
# print(raw.ch_names)
eeg_initial_data = raw.get_data()
print(eeg_initial_data.shape)
raw.plot(n_channels = len(raw.ch_names), duration=20.0, start=0.0)

In [ ]:
!python main.py

In [ ]:
csv_path = 'data/our_output.csv'
edf_path = 'data/our_output.edf'

output = pd.read_csv(csv_path, header=None)
output = output.transpose()

n_channels = output.shape[1]
n_samples = output.shape[0]
sampling_rate = 500

signal_headers = []
for ch in range(n_channels):
    signal_headers.append({
        'label': channel_names[ch],
        'dimension': 'uV',
        'sample_frequency': sampling_rate,
        'physical_min': np.min(output.iloc[:, ch]),
        'physical_max': np.max(output.iloc[:, ch]),
        'digital_min': -32768,
        'digital_max': 32767,
        'transducer': '',
        'prefilter': ''
    })

edf_writer = pyedflib.EdfWriter(edf_path, n_channels=n_channels)
edf_writer.setSignalHeaders(signal_headers)

signals = [output.iloc[:, ch].to_numpy().astype(np.float64) for ch in range(n_channels)]

edf_writer.writeSamples(signals)
edf_writer.close()

print(f"File saved as {edf_path}")

In [ ]:
raw_after = mne.io.read_raw_edf("data/our_output.edf", preload=True)
data_after = raw_after.get_data()
data_after = data_after[:, start_phys:numpoint]
channel_names_after = raw_after.ch_names

eeg_after_data = raw_after.get_data()
print(eeg_after_data.shape)
raw_after.plot(n_channels = len(raw_after.ch_names), duration=20.0, start=0.0)

Visualization by channels

In [59]:
num_of_channels = len(data)
channel_len_before = len(data[0])
channel_len_after = len(data_after[0])

In [ ]:
fig, axes = plt.subplots(num_of_channels, 2, figsize=(10, 2 * num_of_channels))

x_before = np.arange(0, 1, 1.0 / channel_len_before)
x_after = np.arange(0, 1, 1.0 / channel_len_after)

for i in range(num_of_channels):
    ax_before = axes[i, 0] if num_of_channels > 1 else axes[0]
    ax_after = axes[i, 1] if num_of_channels > 1 else axes[1]

    mean_before = np.mean(data[i])
    mean_after = np.mean(data_after[i])

    mode_before = stats.mode(data[i], keepdims=False)
    mode_after = stats.mode(data_after[i], keepdims=False)

    ax_before.plot(x_before, data[i])
    ax_before.set_title(f'Channel {i+1}, mean: {mean_before:.2f}, mode: {mode_before.mode:.2f}')
    
    ax_after.plot(x_after, data_after[i])
    ax_after.set_title(f'Channel {i+1}, mean: {mean_after:.2f}, mode: {mode_after.mode:.2f}')

fig.text(0.25, 0.99, 'Before', ha='center', va='bottom', fontsize=14)
fig.text(0.75, 0.99, 'After', ha='center', va='bottom', fontsize=14)

fig.suptitle('Visualization of our data, EEG graph before and after ART', fontsize=16, y=1.005)

plt.tight_layout()
plt.show()